All cridets [@hidehisaarai1213](https://www.kaggle.com/hidehisaarai1213)

This notebook based on this [Introduction to Sound Event Detection](https://www.kaggle.com/hidehisaarai1213/introduction-to-sound-event-detection)

### Install packages

In [ ]:
local = False
test_only = False


if not local:
    !pip -q install timm
    !pip -q install torchlibrosa
    !pip -q install audiomentations

### import packages

In [ ]:
import os, sys, glob, random, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import librosa, librosa.display
import soundfile as sf

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

from functools import partial
from sklearn import metrics
from sklearn.model_selection import StratifiedKFold
from transformers import get_linear_schedule_with_warmup
from torchlibrosa.stft import Spectrogram, LogmelFilterBank
from torchlibrosa.augmentation import SpecAugmentation

### About Sound Event Detection(SED)

Sound event detection (SED) is the task of detecting the type as well as
the onset and offset times of sound events in audio streams.

In this notebook i will show how to train Sound Event Detection (SED) model with only weak annotation.

In SED task, we need to detect sound events from continuous (long) audio clip, and provide prediction of what sound event exists from when to when.

for more details

-> [Polyphonic Sound Event Detection
with Weak Labeling Paper](http://www.cs.cmu.edu/~yunwang/papers/cmu-thesis.pdf)

-> [Introduction to Sound Event Detection Notebook](https://www.kaggle.com/hidehisaarai1213/introduction-to-sound-event-detection)

### PANN Utils

-> [PANNs repository](https://github.com/qiuqiangkong/audioset_tagging_cnn/)

-> [PANNs paper](https://arxiv.org/abs/1912.10211)

In [ ]:
def init_layer(layer):
    nn.init.xavier_uniform_(layer.weight)

    if hasattr(layer, "bias"):
        if layer.bias is not None:
            layer.bias.data.fill_(0.)


def init_bn(bn):
    bn.bias.data.fill_(0.)
    bn.weight.data.fill_(1.0)


def init_weights(model):
    classname = model.__class__.__name__
    if classname.find("Conv2d") != -1:
        nn.init.xavier_uniform_(model.weight, gain=np.sqrt(2))
        model.bias.data.fill_(0)
    elif classname.find("BatchNorm") != -1:
        model.weight.data.normal_(1.0, 0.02)
        model.bias.data.fill_(0)
    elif classname.find("GRU") != -1:
        for weight in model.parameters():
            if len(weight.size()) > 1:
                nn.init.orghogonal_(weight.data)
    elif classname.find("Linear") != -1:
        model.weight.data.normal_(0, 0.01)
        model.bias.data.zero_()


def interpolate(x: torch.Tensor, ratio: int):
    """Interpolate data in time domain. This is used to compensate the
    resolution reduction in downsampling of a CNN.
    Args:
      x: (batch_size, time_steps, classes_num)
      ratio: int, ratio to interpolate
    Returns:
      upsampled: (batch_size, time_steps * ratio, classes_num)
    """
    (batch_size, time_steps, classes_num) = x.shape
    upsampled = x[:, :, None, :].repeat(1, 1, ratio, 1)
    upsampled = upsampled.reshape(batch_size, time_steps * ratio, classes_num)
    return upsampled


def pad_framewise_output(framewise_output: torch.Tensor, frames_num: int):
    """Pad framewise_output to the same length as input frames. The pad value
    is the same as the value of the last frame.
    Args:
      framewise_output: (batch_size, frames_num, classes_num)
      frames_num: int, number of frames to pad
    Outputs:
      output: (batch_size, frames_num, classes_num)
    """
    pad = framewise_output[:, -1:, :].repeat(
        1, frames_num - framewise_output.shape[1], 1)
    """tensor for padding"""

    output = torch.cat((framewise_output, pad), dim=1)
    """(batch_size, frames_num, classes_num)"""

    return output

### Create Folds

In [ ]:
def create_folds():
    ''' 
    Split kaggle dataset into [5] folds. Re-write the csv with fold information.
    implicit input: kaggle dataset rfcx-species-audio-detection/train_tp.csv
    output: ./train_folds.csv
    '''
    train = pd.read_csv(args.train_tp_csv).sort_values("recording_id")
    
    train_gby = train.groupby("recording_id")[["species_id"]].first().reset_index()
    train_gby = train_gby.sample(frac=1, random_state=args.seed).reset_index(drop=True)
    train_gby.loc[:, 'kfold'] = -1
    
    X = train_gby["recording_id"].values
    y = train_gby["species_id"].values
    
    kfold = StratifiedKFold(n_splits=args.FOLDS)
    for fold, (t_idx, v_idx) in enumerate(kfold.split(X, y)):
        #print("i_idx", t_idx, "v_idx", v_idx)
        train_gby.loc[v_idx, "kfold"] = fold # mark validation set : 0..4
    
    train = train.merge(train_gby[['recording_id', 'kfold']], on="recording_id", how="left")
    print('train.kfold.value_counts:\n{}'.format(train.kfold.value_counts()))
    if os.path.dirname(args.train_csv):
        os.makedirs(os.path.dirname(args.train_csv), exist_ok = True)
    train.to_csv(args.train_csv, index=False)

def create_foldsfp():
    ''' 
    Split kaggle dataset into [5] folds. Re-write the csv with fold information.
    implicit input: kaggle dataset rfcx-species-audio-detection/train_tp.csv
    output: ./train_folds.csv
    '''
    trainfp = pd.read_csv(args.train_fp_csv).sort_values("recording_id")
    
    trainfp_gby = trainfp.groupby("recording_id")[["species_id"]].first().reset_index()
    trainfp_gby = trainfp_gby.sample(frac=1, random_state=args.seed).reset_index(drop=True)
    trainfp_gby.loc[:, 'kfold'] = -1
    
    X = trainfp_gby["recording_id"].values
    y = trainfp_gby["species_id"].values
    
    kfold = StratifiedKFold(n_splits=args.FOLDS)
    # t_idx: array of indexes of the trainset, v_idx: array of testset
    for fold, (t_idx, v_idx) in enumerate(kfold.split(X, y)):
        #print("i_idx", t_idx, "v_idx", v_idx)
        trainfp_gby.loc[v_idx, "kfold"] = fold # mark validation set : 0..4
    
    trainfp = trainfp.merge(trainfp_gby[['recording_id', 'kfold']], on="recording_id", how="left")
    print('trainfp.kfold.value_counts:\n{}'.format(trainfp.kfold.value_counts()))
    if os.path.dirname(args.trainfp_csv):
        os.makedirs(os.path.dirname(args.trainfp_csv), exist_ok = True)
    trainfp.to_csv(args.trainfp_csv, index=False)

### SED Model

1. Model takes raw waveform and converted into log-melspectogram using `torchlibrosa`'s module
2. spectogram converted into 3-channels input for ImageNet pretrain model to extract features from CNN's
3. Although it's downsized through several convolution and pooling layers, 
   the size of it's third dimension and it still contains time information. 
   Each element of this dimension is segment. In SED model, we provide prediction for each of this.

4. This figure gives us an intuitive explanation what is weak annotation and 
   what is strong annotation in terms of sound event detection. For this competition,
   we only have weak annotation (clip level annotation). Therefore, we need to train our SED model in weakly-supervised manner.

5. In weakly-supervised setting, we only have clip-level annotation, therefore we also need to aggregate that in time axis. Hense, we at first put classifier that outputs class existence probability for each time step just after the feature extractor and then aggregate the output of the classifier result in time axis. In this way we can get both clip-level prediction and segment-level prediction (if the time resolution is high, it can be treated as event-level prediction). Then we train it normally by using BCE loss with clip-level prediction and clip-level annotation.

In [ ]:
class Identity(nn.Module):
    def __init__(self):
        super(Identity, self).__init__()
        
    def forward(self, x):
        print('identity module x.shape=', x.shape)
        return x

class AttBlock(nn.Module):
    def __init__(self,in_ft, out_ft):
        super(AttBlock, self).__init__()
        self.in_ft, self.out_ft = in_ft, out_ft
        self.conv1 = nn.Conv2d(
                                in_channels=in_ft,
                                out_channels=out_ft,
                                kernel_size=(1,4),
                                stride=(1,4),
                                padding=0,
                                bias=False)        
        self.conv2 = nn.Conv2d(
                                in_channels=in_ft,
                                out_channels=out_ft,
                                kernel_size=(3,4),
                                stride=(1,4),
                                padding=(1,0),
                                bias=False)        
        init_layer(self.conv1)
        init_layer(self.conv2)
    
    # With attention block
    # def forward(self, x): #[16, 512, 10, 4]
    #     xshape = x.shape
    #     x = x.transpose(2,3) #[16, 512, 4, 10]
    #     x = x.reshape(xshape[0], -1, xshape[2]) #[16, 2048, 10]
    #     x1 = F.max_pool1d(x, kernel_size=3, stride=1, padding=1) #[16, 2048, 10]
    #     x2 = F.avg_pool1d(x, kernel_size=3, stride=1, padding=1) #[16, 2048, 10]
    #     x = x1 + x2 #[16, 2048, 10]
    #     x = x.reshape(xshape[0], xshape[1], xshape[3], xshape[2]) #[16, 512, 4, 10]
    #     x = x.transpose(2,3) #[16, 512, 10, 4]
    
    #     x1 = self.conv1(x) #[16, 24, 10, 1]
    #     # x1 = self.bn1(x1)   #[16, 24, 10, 1]
    #     x1 = x1.squeeze(dim=3) #[16, 24, 10]
    #     x2 = self.conv2(x) #[16, 24, 10, 1]
    #     # x2 = self.bn2(x2)   #[16, 24, 10, 1]
    #     x2 = x2.squeeze(dim=3) #[16, 24, 10]
        
    #     norm_att = torch.softmax(torch.clamp(x1, -10, 10), dim=-1) #[16, 24, 10]
    #     cla = torch.sigmoid(x2) #[16, 24, 10]
    #     x = torch.sum(norm_att * cla, dim=2) #[16, 24]
        
    #     return x

    # Without attention block
    def forward(self, x): #[16, 512, 10, 4]
        # xshape = x.shape
        # x = x.transpose(2,3) #[16, 512, 4, 10]
        # x = x.reshape(xshape[0], -1, xshape[2]) #[16, 2048, 10]
        # x1 = F.max_pool1d(x, kernel_size=3, stride=1, padding=1) #[16, 2048, 10]
        # x2 = F.avg_pool1d(x, kernel_size=3, stride=1, padding=1) #[16, 2048, 10]
        # x = x1 + x2 #[16, 2048, 10]
        # x = x.reshape(xshape[0], xshape[1], xshape[3], xshape[2]) #[16, 512, 4, 10]
        # x = x.transpose(2,3) #[16, 512, 10, 4]
    
        x = self.conv1(x) #[16, 24, 10, 1]
        # x = self.bn1(x)   #[16, 24, 10, 1]
        x = x.squeeze(dim=3) #[16, 24, 10]
        
        x = torch.max(x, dim=2)[0] #[16, 24]
        # x = torch.sum(x, dim=2) #[16, 24] insteqd of max => extremely high loss !
        x = torch.sigmoid(x)
        
        return x
    
class AudioSEDModel(nn.Module):
    def __init__(self, encoder, sample_rate, window_size, hop_size, 
                 mel_bins, fmin, fmax, classes_num):
        super().__init__()

        window = 'hann'
        center = True
        pad_mode = 'reflect'
        ref = 1.0
        amin = 1e-10
        top_db = None

        # Spectrogram extractor
        self.spectrogram_extractor = Spectrogram(n_fft=window_size, hop_length=hop_size, 
            win_length=window_size, window=window, center=center, pad_mode=pad_mode, 
            freeze_parameters=True)

        # Logmel feature extractor
        self.logmel_extractor = LogmelFilterBank(sr=sample_rate, n_fft=window_size, 
            n_mels=mel_bins, fmin=fmin, fmax=fmax, ref=ref, amin=amin, top_db=top_db, 
            freeze_parameters=True)

        # self.bn0 = nn.BatchNorm2d(mel_bins)
        self.bn0 = nn.BatchNorm2d(1)

        # Spec augmenter
        self.spec_augmenter = SpecAugmentation(time_drop_width=64, time_stripes_num=2, 
            freq_drop_width=8, freq_stripes_num=2)
                
        # Model Encoder
        self.encoder = torchvision.models.resnet18(pretrained=True)
        self.enc_out_feat = self.encoder.fc.in_features
        
        # feature extraction ot finetuning, applied to pretrained encoder only
        for param in self.encoder.parameters(): 
                param.requires_grad = args.finetune
            
        # class re-assignment : #[16, 512] -> #[16, 24]

        # Replace pre-trained encoder's last layer
        self.encoder.avgpool = AttBlock(self.enc_out_feat, classes_num)
        self.encoder.fc = nn.Sequential()
        
        self.init_weight()

    def init_weight(self):
        init_bn(self.bn0)
        
    def forward(self, input):
        """Input : (batch_size, data_length) : [16, 160000] """

        x = self.spectrogram_extractor(input) # [16, 1, 313, 2049]
        # batch_size x 1 x time_steps x freq_bins
        x = self.logmel_extractor(x) # [16, 1, 313, 128]
        # batch_size x 1 x time_steps x mel_bins
        bs, _, frames_num, _ = x.shape # 16, 313

        if self.training:
            x = self.spec_augmenter(x)

        # x = x.transpose(1, 3) # [16, 128, 313, 1]
        x = self.bn0(x)
        # x = x.transpose(1, 3) # [16, 1, 313, 128]
        
        # features extraction
        
        x = x.expand(x.shape[0], 3, x.shape[2], x.shape[3]) # [16, 3, 313, 128]
        # Intermediate utput shape (batch size, channels, time, frequency)
        # Final output shape (batch size, classes_num)
        x = self.encoder(x) #[16, 24]
        #x = F.dropout(x, p=0.5, training=self.training)
        
        return x

### Dataset

In [ ]:
def crop_or_pad(y, sr, period, record, mode="train"):
    len_y = len(y)
    effective_length = sr * period
    rint = np.random.randint(len(record['t_min']))
    sample_start = record['t_min'][rint] * sr
    sample_end = record['t_max'][rint] * sr
    if len_y > effective_length:
        # Positioning sound slice
        center = np.round((sample_start + sample_end) / 2)
        beginning = center - effective_length / 2
        if beginning < 0:
            beginning = 0
        beginning = np.random.randint(beginning, center)
        ending = beginning + effective_length
        if ending > len_y:
            ending = len_y
        beginning = ending - effective_length
        y = y[beginning:ending].astype(np.float32)
    else:
        y = y.astype(np.float32)
        beginning = 0
        ending = effective_length


    beginning_time = beginning / sr
    ending_time = ending / sr
    label = np.zeros(24, dtype='f')

    for i in range(len(record['t_min'])):
        if (record['t_min'][i] <= ending_time) and (record['t_max'][i] >= beginning_time):
            label[record['species_id'][i]] = 1
    
    return y, label

In [ ]:
class SedDataset:
    def __init__(self, df, period=10, stride=5, audio_transform=None, data_path="train", mode="train"):

        self.period = period
        self.stride = stride
        self.audio_transform = audio_transform
        self.data_path = data_path
        self.mode = mode

        self.df = df.groupby("recording_id").agg(lambda x: list(x)).reset_index()
        
        self.species = {k:[] for k in range(24)}
        if (mode == 'train'):
            for i in range(self.df.shape[0]):
                record = self.df.iloc[i]
                for k in record["species_id"]:
                    self.species[k].append(i)

        
    def getSpecies(self, target): # (bs, nrOfClass)
        species = target.view(-1)
        _, idx = torch.max(target, dim=1)
        data = []
        for b in range(target.shape[0]):
            rec_list = self.species[idx[b].item()]
            rint = np.random.randint(len(rec_list))
            rec = rec_list[rint]
            data.append(self.__getitem__(rec))
        return data

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        record = self.df.iloc[idx]

        files = glob.glob(f"{self.data_path}/{record['recording_id']}.*")
        if len(files) == 0:
            raise Exception(f"{self.data_path}/{record['recording_id']}.*")
        # y, sr = sf.read(f"{self.data_path}/{record['recording_id']}.flac")
        y, sr = sf.read(files[0])
        
        if self.mode != "test":
            y, label = crop_or_pad(y, sr, period=self.period, record=record, mode=self.mode)

            if self.audio_transform:
                y = self.audio_transform(samples=y, sample_rate=sr)
        else:
            y_ = []
            i = 0
            effective_length = self.period * sr
            stride = self.stride * sr
            # y = np.stack([y[i:i+effective_length].astype(np.float32) for i in range(0, 60*sr+stride-effective_length, stride)])
            for i in range(0, 60*sr-effective_length+1, stride):
                y_.append(y[i:i+effective_length].astype(np.float32))
            y = np.stack(y_)
            label = np.zeros(24, dtype='f')
        
        return {
            "image" : y,
            "target" : label,
            "id" : record['recording_id']
        }

### Augmentations

In [ ]:
import audiomentations as AA

train_audio_transform = AA.Compose([
    AA.AddGaussianNoise(p=0.5),
    AA.AddGaussianSNR(p=0.5),
    #AA.AddBackgroundNoise("../input/train_audio/", p=1)
    #AA.AddImpulseResponse(p=0.1),
    #AA.AddShortNoises("../input/train_audio/", p=1)
    #AA.FrequencyMask(min_frequency_band=0.0,  max_frequency_band=0.2, p=0.1),
    #AA.TimeMask(min_band_part=0.0, max_band_part=0.2, p=0.1),
    #AA.PitchShift(min_semitones=-0.5, max_semitones=0.5, p=0.1),
    #AA.Shift(p=0.1),
    #AA.Normalize(p=0.1),
    #AA.ClippingDistortion(min_percentile_threshold=0, max_percentile_threshold=1, p=0.05),
    #AA.PolarityInversion(p=0.05),
    #AA.Gain(p=0.2)
])

### Utils

In [ ]:
def _lwlrap_sklearn(truth, scores):
    """Reference implementation from https://colab.research.google.com/drive/1AgPdhSp7ttY18O3fEoHOQKlt_3HJDLi8"""
    sample_weight = np.sum(truth > 0, axis=1) # truth[144,24], 144 samples = 9(batches)x16(batchsz)
    nonzero_weight_sample_indices = np.flatnonzero(sample_weight > 0)
    overall_lwlrap = metrics.label_ranking_average_precision_score(
        truth[nonzero_weight_sample_indices, :] > 0,                 # (144, 24)
        scores[nonzero_weight_sample_indices, :],                    # (144, 24)
        sample_weight=sample_weight[nonzero_weight_sample_indices])  # (144,)
    return overall_lwlrap

class AverageMeter(object):
    """Computes and stores the average and current value"""

    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

class MetricMeter(object):
    def __init__(self):
        self.reset()
    
    def reset(self):
        self.y_true = []
        self.y_pred = []
    
    def update(self, y_true, y_pred):
        # print(f'torch.max(y_true)={torch.max(y_true)} shape{y_true.shape} tot-len {len(self.y_true)}')
        self.y_true.extend(y_true.cpu().detach().numpy().tolist())
        self.y_pred.extend(y_pred.cpu().detach().numpy().tolist())

    @property
    def avg(self):
        #score_class, weight = lwlrap(np.array(self.y_true), np.array(self.y_pred))
        self.score = _lwlrap_sklearn(np.array(self.y_true), np.array(self.y_pred)) #(score_class * weight).sum()
        return {
            "lwlrap" : self.score
        }

def seed_everithing(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

def reseedOnResume():
    if args.pretrain_weights and os.path.exists(args.pretrain_weights):        
        checkpoint = torch.load(args.pretrain_weights, map_location=args.device)
        if type(checkpoint) is dict and 'seed' in checkpoint:
                new_seed = checkpoint['seed'] + 1
                print(f"-- Re-seed -- : old {args.seed} new {new_seed}")
                args.seed = new_seed

### Losses

In [ ]:
from torch.nn import BCEWithLogitsLoss, CrossEntropyLoss

class PANNsLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCELoss()

    def forward(self, input, target):
        input = torch.clamp(input, 0.0, 0.99999)
        target = target.float()
        return self.bce(input, target)

### Functions

In [ ]:
# def train_epoch(args, model, loader, criterion, optimizer, scheduler, epoch, fpDs):
#     losses = AverageMeter()
#     scores = MetricMeter()

#     model.train()
#     for i, sample in enumerate(loader):
#         # -- TP --
#         optimizer.zero_grad()
#         input = sample['image'].to(args.device)
#         target = sample['target'].to(args.device)
#         output = model(input)
#         loss = criterion(output, target)
#         # loss.backward()      
#         # optimizer.step()
#         # -- FP --
#         optimizer.zero_grad()
#         fp_sample = fpDs.getSpecies(target)
#         input = [fp_sample[i]['image'] for i in range(len(fp_sample))]
#         input = torch.tensor(input).to(args.device)
#         target = [fp_sample[i]['target'] for i in range(len(fp_sample))]
#         target = torch.tensor(target).to(args.device)
#         output = model(input)
#         index_fp = target == 1
#         index_nonfp = target == 0
#         target[index_fp] = 0
#         target[index_nonfp] = 0.2      
#         loss2 = criterion(output, target) / 10.
#         # loss.backward()
#         # optimizer.step()
#         # -- av loss of TP + FP
#         loss = loss + loss2
#         loss.backward()
#         optimizer.step()
        
#         if scheduler and args.step_scheduler:
#             scheduler.step()

#         bs = input.size(0)
#         scores.update(target, output)
#         losses.update(loss.item(), bs)

#     # print(f"Train E:{epoch} - Loss{losses.avg:0.4f}")
#     return scores.avg, losses.avg

def train_epoch(args, model, loader, criterion, optimizer, scheduler, epoch, fpDs):
    losses = AverageMeter()
    scores = MetricMeter()

    model.train()
    for i, sample in enumerate(loader):
        optimizer.zero_grad()
        criterion = nn.BCELoss(reduction='none')
        # -- TP --
        input = sample['image'].to(args.device)
        target = sample['target'].to(args.device)
        output = model(input)
        loss = criterion(output, target)
        loss = torch.mean(loss)
        loss.backward()      
        # -- FP --
        fp_sample = fpDs.getSpecies(target)
        fp_input = torch.tensor([fp_sample[i]['image'] for i in range(len(fp_sample))])
        fp_target = torch.tensor([fp_sample[i]['target'] for i in range(len(fp_sample))])
        fp_input = fp_input.to(args.device)
        index_fp = (fp_target == 1).to(args.device)
        index_nonfp = (fp_target == 0).to(args.device)
        fp_output = model(fp_input)
        fp_target = (fp_target * 0).to(args.device)
        fp_loss = criterion(fp_output[index_fp], fp_target[index_fp])
        fp_loss = torch.mean(fp_loss)*0.1
        fp_loss.backward()        
        # -- apply accumulated grad     
        optimizer.step()
        
        if scheduler and args.step_scheduler:
            scheduler.step()

        bs = input.size(0)
        scores.update(target, output)
        losses.update(loss.item(), bs)

    # print(f"Train E:{epoch} - Loss{losses.avg:0.4f}")
    return scores.avg, losses.avg
        
def valid_epoch(args, model, loader, criterion, epoch):
    losses = AverageMeter()
    scores = MetricMeter()
    model.eval()
    with torch.no_grad():
        for i, sample in enumerate(loader):
            input = sample['image'].to(args.device)
            target = sample['target'].to(args.device)
            output = model(input)
            loss = criterion(output, target)

            bs = input.size(0)
            scores.update(target, output)
            losses.update(loss.item(), bs)
    # print(f"Valid E:{epoch} - Loss:{losses.avg:0.4f}")
    return scores.avg, losses.avg

def test_epoch(args, model, loader):
    model.eval()
    pred_list = []
    id_list = []
    with torch.no_grad():
        for i, sample in enumerate(loader):
            input = sample["image"].to(args.device)
            bs, seq, w = input.shape
            input = input.reshape(bs*seq, w)
            id = sample["id"]
            output = model(input)
            output = output.reshape(bs, seq, -1)
            output = torch.sum(output, dim=1) / seq
            #output, _ = torch.max(output, dim=1)
            output = output.cpu().detach().numpy().tolist()
            pred_list.extend(output)
            id_list.extend(id)
    
    return pred_list, id_list

### Main Function

In [ ]:
def test_one_model(modelNr, modelPath=None):
    if not modelPath:
        modelPath=os.path.join(args.save_path, f'fold-{modelNr}.bin')
    if not os.path.exists(modelPath):
        print('test_one_model model does not exists:', modelPath)
        return

    sub_df = pd.read_csv(args.sample_submission_csv)
    if args.DEBUG:
        sub_df = sub_df.sample(20)

    test_dataset = SedDataset(
        df = sub_df,
        period=args.period,
        stride=5,
        audio_transform=None,
        data_path=args.test_data_path,
        mode="test"
    )

    test_loader = torch.utils.data.DataLoader(
        test_dataset,
        batch_size=args.batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=args.num_workers
    )

    model = AudioSEDModel(**args.model_param)
    model = model.to(args.device)
    
    checkpoint = torch.load(modelPath, map_location=args.device)
    if type(checkpoint) is dict:
        model.load_state_dict(checkpoint['state'], strict=False)
    else:
        model.load_state_dict(checkpoint, strict=False) 
    # model.load_state_dict(torch.load(modelPath, map_location=args.device))
    model = model.to(args.device)

    target_cols = sub_df.columns[1:].values.tolist()
    test_pred, ids = test_epoch(args, model, test_loader)
    print('(n recordings, n species) =', # (nb of recordings, 24-species)
          np.array(test_pred).shape, ', max', np.max(np.array(test_pred)))

    test_pred_df = pd.DataFrame({
        "recording_id" : sub_df.recording_id.values
    })
    test_pred_df[target_cols] = test_pred
    test_pred_df.to_csv(os.path.join(args.save_path, f"fold-{modelNr}-submission.csv"), index=False)
    print(os.path.join(args.save_path, f"fold-{modelNr}-submission.csv"))

In [ ]:
import re
def main_test():
    args.save_path = os.path.join(args.output_dir, args.exp_name)
    os.makedirs(args.save_path, exist_ok=True)    

    files = glob.glob(f"{args.save_path}/fold-*.bin")
    files.extend(glob.glob(f"{args.input_dir}/fold-*.bin"))
    print(f"main_test : {len(files)} models found")
    for f in files:
        print(f'model file : {f}')
        modelNr = re.compile('.*fold-([0-9]+).bin').match(f).group(1)
        print(f'main_test : model {f}')
        test_one_model(modelNr, f)

    # Ensemble multiple fold-n-submission.csv into submission.csv
    print('Ensemble : ')
    fsubs = glob.glob(os.path.join(args.save_path, "fold-*-submission.csv"))
    sub = None
    sub_df = None
    cnt = 0
    for fsub in fsubs:
        print('sub = ',fsub)
        sub_df = pd.read_csv(fsub)
        if cnt == 0:
            sub = sub_df.iloc[:, 1:]
        else:
            sub += sub_df.iloc[:, 1:]
        cnt += 1
    if cnt > 0:
        sub_df.iloc[:, 1:] = sub / cnt
        sub_df.to_csv("submission.csv", index=False)
        print('Output in submission.csv')
    else:
        print('Ensembling not done')

In [ ]:
def main_CV(fold):
    best_lwlrap = -np.inf
    early_stop_count = 0

    args.save_path = os.path.join(args.output_dir, args.exp_name)
    os.makedirs(args.save_path, exist_ok=True)

    model = AudioSEDModel(**args.model_param)
    model = model.to(args.device)
    
    if args.pretrain_weights and os.path.exists(args.pretrain_weights):
        print("-- loading pretrain weights -- : ", args.pretrain_weights)
        checkpoint = torch.load(args.pretrain_weights, map_location=args.device)
        if type(checkpoint) is dict:
            model.load_state_dict(checkpoint['state'], strict=False)
            if 'best' in checkpoint:
                best_lwlrap = checkpoint['best']
            print("previous best_lwlrap = ", best_lwlrap)
        else:
            model.load_state_dict(checkpoint, strict=False)
        model = model.to(args.device)

    criterion = PANNsLoss() #BCEWithLogitsLoss() #MaskedBCEWithLogitsLoss() #BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr,weight_decay=0.005)

    train_df = pd.read_csv(args.train_csv)
    if args.DEBUG:
        train_df = train_df.sample(args.DBG_TRAIN_LEN)
    train_fold = train_df[train_df.kfold != (fold)]
    valid_fold = train_df[train_df.kfold == (fold)]

    train_dataset = SedDataset(
        df = train_fold,
        period=args.period,
        audio_transform=train_audio_transform,
        data_path=args.train_data_path,
        mode="train"
    )

    valid_dataset = SedDataset(
        df = valid_fold,
        period=args.period,
        audio_transform=None,
        data_path=args.train_data_path,
        mode="valid"
    )

    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=args.batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=args.num_workers
    )

    valid_loader = torch.utils.data.DataLoader(
        valid_dataset,
        batch_size=args.batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=args.num_workers
    )

    trainfp_df = pd.read_csv(args.trainfp_csv)
    if args.DEBUG:
        trainfp_df = trainfp_df.sample(args.DBG_TRAIN_LEN)
    trainfp_fold = trainfp_df[trainfp_df.kfold != (fold)]
    validfp_fold = trainfp_df[trainfp_df.kfold == (fold)]

    trainfp_dataset = SedDataset(
        df = trainfp_fold,
        period=args.period,
        audio_transform=train_audio_transform,
        data_path=args.train_data_path,
        mode="train"
    )


    num_train_steps = int(len(train_loader) * args.epochs)
    num_warmup_steps = int(0.1 * args.epochs * len(train_loader))
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=num_train_steps)

    for epoch in range(args.start_epcoh, args.epochs):
        
        train_avg, train_loss = train_epoch(args, model, train_loader, criterion, optimizer, scheduler, epoch, trainfp_dataset)
        valid_avg, valid_loss = valid_epoch(args, model, valid_loader, criterion, epoch)

        content = f"""
                {time.ctime()} \n
                Fold:{fold}, Epoch:{epoch}, lr:{optimizer.param_groups[0]['lr']:.7}\n
                Train Loss:{train_loss:0.4f} - LWLRAP:{train_avg['lwlrap']:0.4f}\n
                Valid Loss:{valid_loss:0.4f} - LWLRAP:{valid_avg['lwlrap']:0.4f}\n
        """
        print(content)
        with open(f'{args.save_path}/log_{args.exp_name}.txt', 'a') as appender:
            appender.write(content+'\n')
        
        if valid_avg['lwlrap'] > best_lwlrap:
            print(f"########## >>>>>>>> Model Improved From {best_lwlrap} ----> {valid_avg['lwlrap']}")
            best_lwlrap = valid_avg['lwlrap']
            early_stop_count = 0
            torch.save({'state': model.state_dict(), 'best': best_lwlrap, 'seed': args.seed},
                       os.path.join(args.save_path, f'fold-{fold}.bin'))
        else:
            early_stop_count += 1

        # ---------------------
        if args.early_stop == early_stop_count:
            print("\n $$$ ---? Ohoo.... we reached early stoping count :", early_stop_count)
            break

        if args.epoch_scheduler:
            scheduler.step()
        
    torch.save({'state': model.state_dict(), 'best': best_lwlrap, 'seed': args.seed},
               os.path.join(args.save_path, f'fold-9999{fold}.bin'))

### Config

In [ ]:
class args:
    
    FOLDS = 5
    if (local):
        DEBUG = True
        train_tp_csv = "C:/kaggle/train_tp.csv"
        train_fp_csv = "C:/kaggle/train_fp.csv"
        sample_submission_csv = "C:/kaggle/sample_submission.csv"
        train_csv = "C:/kaggle/work/train_folds.csv"
        trainfp_csv = "C:/kaggle/work/trainfp_folds.csv"
        test_csv = "C:/kaggle/work/test_df.csv"
        output_dir = "C:/kaggle/work/weights"
        input_dir = "C:/kaggle/modelrfcx"
        train_data_path = "C:/kaggle/train"
        test_data_path  = "C:/kaggle/test"
        num_workers = 0
        epochs = 6 if DEBUG else 50
        DBG_TRAIN_LEN = 200
        pretrain_weights = "C:/kaggle/modelrfcx/latest.bin"
    else:
        DEBUG = False
        train_tp_csv = "../input/rfcx-species-audio-detection/train_tp.csv"
        train_fp_csv = "../input/rfcx-species-audio-detection/train_fp.csv"
        sample_submission_csv = "../input/rfcx-species-audio-detection/sample_submission.csv"
        train_csv = "train_folds.csv"
        trainfp_csv = "trainfp_folds.csv"
        test_csv = "test_df.csv"
        output_dir = "weights"
        input_dir = "../input/modelrfcx"
        train_data_path = "../input/rfcx-species-audio-detection/train"
        test_data_path = "../input/rfcx-species-audio-detection/test"
        num_workers = 4
        epochs = 1 if DEBUG else 50
        DBG_TRAIN_LEN = 400
        pretrain_weights = "../input/modelrfcx/latest.bin"
    
    exp_name = "SED_E0_5F_BASE"
    # pretrain_weights = None 
    model_param = {
        'encoder' : 'resnet18', #'tf_efficientnet_b0_ns',
        'sample_rate': 16000 if local else 48000,
        'window_size' : 512 * 2, # 512
        'hop_size' : 512, 
        'mel_bins' : 128, 
        'fmin' : 0,
        'fmax' : 16000//2 if local else 48000//2, # 48000 // 2,
        'classes_num' : 24
    }
    finetune = True
    period = 10
    seed = 42
    start_epcoh = 0 
    lr = 1e-3
    batch_size = 16
    early_stop = 150
    step_scheduler = True
    epoch_scheduler = False

    device = ('cuda' if torch.cuda.is_available() else 'cpu')

### train folds

In [ ]:
reseedOnResume()
if local:
    os.environ['TORCH_HOME'] = os.path.join(args.output_dir, 'resnet')
    
if test_only:
    seed_everithing(args.seed)
    main_test()
else:
    seed_everithing(args.seed)
    create_folds()
    create_foldsfp()
    seed_everithing(args.seed)
    if (local):
        main_CV(0)
    else:
        main_CV(0)
        # main_CV(1)
        # main_CV(2)
        #main_train()
    
    seed_everithing(args.seed)
    # main_test()
    test_one_model(0)
    test_one_model(99990)